# pymfx — Build a .mfx file from scratch

This notebook shows how to construct a complete `.mfx` file from raw Python data
without starting from an existing file:

1. Build all objects manually
2. Add a trajectory with real GPS points
3. Add events
4. Compute the bounding box and index
5. Add a weather extension
6. Write and validate the result

In [ ]:
import sys
sys.path.insert(0, '..')  # if running from notebooks/ folder

import uuid
from datetime import datetime, timezone, timedelta

import pymfx
from pymfx.models import (
    MfxFile, Meta, Trajectory, TrajectoryPoint, SchemaField,
    Events, Event, Index, Extension
)

print(f"pymfx {pymfx.__version__}")

## 1. Define mission parameters

In [ ]:
MISSION_ID = f"uuid:{uuid.uuid4()}"
DATE_START = "2026-04-01T09:00:00Z"
DATE_END   = "2026-04-01T09:08:20Z"

print(f"Mission ID : {MISSION_ID}")

## 2. Build the [meta] section

In [ ]:
meta = Meta(
    id=MISSION_ID,
    drone_id="DJI-Mavic3-SN112233",
    drone_type="multirotor",
    manufacturer="DJI",
    pilot_id="FR-PILOT-0042",
    date_start=DATE_START,
    date_end=DATE_END,
    duration_s=500,
    status="complete",
    application="coastal_survey",
    location="Biarritz, France",
    crs="WGS84",
    altitude_ref="MSL",
    sensors=["rgb", "thermal"],
    data_level="raw",
    processing_tools="none",
    producer="pymfx",
    producer_version="1.0.0",
    source_format="native",
    license="CC-BY-4.0",
    contact="survey@coastal.fr",
)

print(f"Meta built for drone: {meta.drone_id}")

## 3. Build trajectory points

In [ ]:
# Simulate a simple survey grid over the coast of Biarritz
# 10 Hz, 50 points
import math

BASE_LAT = 43.4832
BASE_LON = -1.5586
FREQUENCY = 10

raw_points = []
for i in range(50):
    t = round(i / FREQUENCY, 3)
    lat = BASE_LAT + (i * 0.00005)
    lon = BASE_LON + math.sin(i * 0.3) * 0.0003
    alt = 50.0 + math.sin(i * 0.2) * 2.0  # 48–52m
    speed = 8.0 + math.cos(i * 0.15) * 1.5
    raw_points.append((t, round(lat, 6), round(lon, 6), round(alt, 1), round(speed, 1)))

print(f"Generated {len(raw_points)} points")
print(f"First: {raw_points[0]}")
print(f"Last:  {raw_points[-1]}")

In [ ]:
# Build SchemaField objects
schema_fields = [
    SchemaField(name="t",        type="float",   constraints=["no_null"]),
    SchemaField(name="lat",      type="float",   constraints=["range=-90..90", "no_null"]),
    SchemaField(name="lon",      type="float",   constraints=["range=-180..180", "no_null"]),
    SchemaField(name="alt_m",    type="float32", constraints=["range=0..10000"]),
    SchemaField(name="speed_ms", type="float32", constraints=["range=0..200"]),
]

# Build raw lines (pipe-separated) and TrajectoryPoint objects
points = []
raw_lines = []

for t, lat, lon, alt, speed in raw_points:
    raw_lines.append(f"{t:.3f} | {lat} | {lon} | {alt} | {speed}")
    points.append(TrajectoryPoint(t=t, lat=lat, lon=lon, alt_m=alt, speed_ms=speed))

print(f"Raw line example: {raw_lines[0]}")

In [ ]:
from pymfx import compute_checksum

traj_checksum = compute_checksum(raw_lines)
print(f"Trajectory checksum: {traj_checksum}")

trajectory = Trajectory(
    frequency_hz=FREQUENCY,
    schema_fields=schema_fields,
    points=points,
    checksum=traj_checksum,
    raw_lines=raw_lines,
)

## 4. Build events

In [ ]:
event_schema = [
    SchemaField(name="t",        type="float", constraints=["no_null"]),
    SchemaField(name="type",     type="str",   constraints=["enum=[takeoff,landing,waypoint,anomaly,rtl,abort]"]),
    SchemaField(name="severity", type="str",   constraints=["enum=[info,warning,critical]"]),
    SchemaField(name="detail",   type="str"),
]

event_list = [
    Event(t=0.0,                 type="takeoff",  severity="info",    detail="nominal"),
    Event(t=2.4,                 type="waypoint", severity="info",    detail="wp1"),
    Event(t=3.1,                 type="anomaly",  severity="warning", detail="wind_gust"),
    Event(t=round(49/FREQUENCY, 3), type="landing",  severity="info",    detail="nominal"),
]

event_raw_lines = []
for e in event_list:
    detail = f'"{e.detail}"' if ' ' in e.detail else e.detail
    event_raw_lines.append(f"{e.t:.3f} | {e.type} | {e.severity} | {detail}")

events_checksum = compute_checksum(event_raw_lines)

events = Events(
    schema_fields=event_schema,
    events=event_list,
    checksum=events_checksum,
    raw_lines=event_raw_lines,
)

print(f"{len(event_list)} events, checksum: {events_checksum}")

## 5. Compute bounding box and build index

In [ ]:
lats = [p.lat for p in points]
lons = [p.lon for p in points]

bbox = (
    round(min(lons), 6),  # lon_min
    round(min(lats), 6),  # lat_min
    round(max(lons), 6),  # lon_max
    round(max(lats), 6),  # lat_max
)

# Count warning/critical events
anomaly_count = sum(1 for e in event_list if e.severity in ('warning', 'critical'))

index = Index(bbox=bbox, anomalies=anomaly_count)

print(f"bbox      : {bbox}")
print(f"anomalies : {anomaly_count}")

## 6. Add a weather extension

In [ ]:
weather_ext = Extension(
    name="x_weather",
    fields={
        "wind_ms":       4.2,
        "wind_dir_deg":  215,
        "temperature_c": 17.5,
        "humidity_pct":  68.0,
        "pressure_hpa":  1013.2,
    }
)

print(f"Extension: {weather_ext.name} — {len(weather_ext.fields)} fields")

## 7. Assemble and write the MfxFile

In [ ]:
mfx = MfxFile(
    version="1.0",
    encoding="UTF-8",
    meta=meta,
    trajectory=trajectory,
    events=events,
    index=index,
    extensions=[weather_ext],
)

# Write — checksums will be recomputed automatically
output = pymfx.write(mfx)

print(f"Output size: {len(output)} chars")
print()
# Print the first 1200 chars
print(output[:1200])

In [ ]:
# Print the full file
print(output)

## 8. Validate the generated file

In [ ]:
mfx_reloaded = pymfx.parse(output)
result = pymfx.validate(mfx_reloaded, raw_text=output)
print(result)

## 9. Save to disk

In [ ]:
pymfx.write(mfx, dest="../tests/biarritz_coastal_survey.mfx")
print("Saved to tests/biarritz_coastal_survey.mfx")

In [ ]:
# Verify from disk
mfx_disk = pymfx.parse("../tests/biarritz_coastal_survey.mfx")
print(f"Points loaded from disk: {len(mfx_disk.trajectory.points)}")
print(f"Events loaded from disk: {len(mfx_disk.events.events)}")
print(f"Extension loaded       : {mfx_disk.extensions[0].name}")
result_disk = pymfx.validate(mfx_disk, raw_text=open('../tests/biarritz_coastal_survey.mfx').read())
print(result_disk)